# Fresh NHAMCS data-to-simulation handoff
This notebook inspects the external dataset, runs the standalone exporter and tests, and reconciles the exported counts. No prior implementation is imported. Execution evidence does not determine competition eligibility. Run from the event_nhamcs directory using the intended Python environment.

In [ ]:
import os, sys, platform, json, hashlib, subprocess
from pathlib import Path
import pandas as pd
print({"architecture": platform.machine(), "python": platform.python_version(), "pandas": pd.__version__})
print("Expected LinuxONE environment:", sys.executable.startswith("/home/linux1/jupyter-env/"))
root = Path.cwd()
assert (root / "export_parameters.py").is_file(), "Run from event_nhamcs"
remote_input = root.parent / "nhamcs_data_2018_22.csv"
local_input = root.parent.parent / "ed-surge-simulator/data/raw/nhamcs/nhamcs_data_2018_22.csv"
source = Path(os.environ.get("NHAMCS_INPUT", str(remote_input if remote_input.exists() else local_input)))
assert source.is_file()
source_hash = hashlib.sha256(source.read_bytes()).hexdigest()
print({"input_file": source.name, "sha256": source_hash})

## Inspect before transforming
Pandas inferred types are descriptive. The exporter parses original CSV tokens and does not use inferred floats for HHMM. No clinical field cleaning is performed.

In [ ]:
raw = pd.read_csv(source, dtype=str, keep_default_na=False)
inferred = pd.read_csv(source)
print("Shape:", raw.shape)
print(pd.DataFrame({"inferred_dtype": inferred.dtypes.astype(str), "blank_cells": raw.apply(lambda s: s.str.strip().eq("").sum())}).to_string())
print("Exact duplicate excess rows:", int(raw.duplicated().sum()))
print("Arrival sample tokens:", raw["arrival_time"].head(10).tolist())

## Export and independently validate
The CLI runs fresh synthetic tests and independent raw-data reconciliation, then records sanitized evidence. Reference checking is bound to the known source SHA-256.

In [ ]:
command = [sys.executable, "-B", "export_parameters.py", "--input", str(source), "--output", "outputs/params.json", "--verify-reference", "--evidence"]
completed = subprocess.run(command, capture_output=True, text=True, check=True)
print(completed.stdout)
params = json.loads(Path("outputs/params.json").read_text())
evidence = json.loads(Path("outputs/execution_evidence.json").read_text())
assert evidence["source_unchanged"] and evidence["repeat_export_identical"]
assert all(evidence["reference_checks"].values())
assert hashlib.sha256(source.read_bytes()).hexdigest() == source_hash
print(json.dumps(evidence, indent=2))

In [ ]:
print(pd.DataFrame({"hour": params["hourly_arrivals"]["hours"], "count": params["hourly_arrivals"]["counts"], "probability": params["hourly_arrivals"]["proportions"]}).to_string(index=False))
print(pd.DataFrame({"triage_code": params["triage"]["codes"], "count": params["triage"]["counts"], "probability": params["triage"]["proportions"]}).to_string(index=False))
print(json.dumps(params["unavailable_parameters"], indent=2))

## Handoff limits
These are pooled unweighted sample distributions, not Houston arrival rates. The Simulation Team must separately provide baseline volume, service times, admission, boarding and capacity assumptions. Weekday labels remain unverified. The compatibility objects do not authorize replacement of the existing PATWT-weighted baseline. Use ED_SCENARIO_PARAMS only after team acceptance. No remote execution is implied by an unexecuted notebook.